In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import re
from datetime import datetime
import yfinance as yf
import time

In [2]:
def get_ipos() -> pd.DataFrame:
    """
    Fetch IPO data for the given year from stockanalysis.com.
    """
    url = f"https://stockanalysis.com/ipos/withdrawn/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found.")

        return tables[0]

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [3]:
ipos_withdrawals = get_ipos()
ipos_withdrawals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Withdrawn Date  146 non-null    object
 1   Symbol          146 non-null    object
 2   Company Name    146 non-null    object
 3   Price Range     146 non-null    object
 4   Shares Offered  146 non-null    object
dtypes: object(5)
memory usage: 5.8+ KB


In [4]:
def classify_company(name):
    name = str(name).lower()
    words = re.findall(r'\b\w+\b', name)  # wyłapuje tylko czyste słowa, ignorując przecinki/kropki

    if 'acquisition' in words and ('corp' in words or 'corporation' in words):
        return 'Acq.Corp'
    elif 'inc' in words or 'incorporated' in words:
        return 'Inc'
    elif 'group' in words:
        return 'Group'
    elif 'ltd' in words or 'limited' in words:
        return 'Limited'
    elif 'holdings' in words:
        return 'Holdings'
    else:
        return 'Other'

In [5]:
ipos_withdrawals['Company Class'] = ipos_withdrawals['Company Name'].apply(classify_company)

In [6]:
ipos_withdrawals

,Withdrawn Date,Symbol,Company Name,Price Range,Shares Offered,Company Class
0,"Sep 25, 2026",HNUC,Holtec Nuclear Corporation,$15.00 - $18.00,50000000,Other
1,"Sep 18, 2026",GEBE,GEBE Environmental Technology Limited,$4.00 - $6.00,3750000,Limited
2,"Sep 18, 2026",RSHL,Rise Smart Group Holdings Limited,$4.00,2000000,Group
3,"Sep 17, 2026",CCLV,"C2 Capital Group, Inc.",$4.00 - $5.00,3750000,Inc
4,"Sep 15, 2026",FCHS,"First Choice Healthcare Solutions, Inc.",$5.00,3800000,Inc
...,...,...,...,...,...,...
141,"Feb 8, 2023",SBP,"Specialty Building Products, Inc.",-,-,Inc
142,"Jan 9, 2023",MFRM,Mattress Firm Group Inc.,-,-,Inc
143,"Sep 2, 2022",CHO,Chobani Inc.,-,-,Inc
144,"Aug 8, 2022",ROTK,"Rotech Healthcare Holdings, Inc.",-,-,Inc


In [7]:
def parse_avg_price(price_range):
    if not isinstance(price_range, str) or '-' in price_range.strip() and price_range.strip() == '-':
        return None

    # Pull out numbers 
    numbers = re.findall(r'\d+(?:\.\d+)?', price_range)

    if not numbers:
        return None
    numbers = list(map(float, numbers))
    return sum(numbers) / len(numbers)

In [8]:
ipos_withdrawals['Avg. price'] = ipos_withdrawals['Price Range'].apply(parse_avg_price)

In [9]:
ipos_withdrawals

,Withdrawn Date,Symbol,Company Name,Price Range,Shares Offered,Company Class,Avg. price
0,"Sep 25, 2026",HNUC,Holtec Nuclear Corporation,$15.00 - $18.00,50000000,Other,16.5
1,"Sep 18, 2026",GEBE,GEBE Environmental Technology Limited,$4.00 - $6.00,3750000,Limited,5.0
2,"Sep 18, 2026",RSHL,Rise Smart Group Holdings Limited,$4.00,2000000,Group,4.0
3,"Sep 17, 2026",CCLV,"C2 Capital Group, Inc.",$4.00 - $5.00,3750000,Inc,4.5
4,"Sep 15, 2026",FCHS,"First Choice Healthcare Solutions, Inc.",$5.00,3800000,Inc,5.0
...,...,...,...,...,...,...,...
141,"Feb 8, 2023",SBP,"Specialty Building Products, Inc.",-,-,Inc,NaN
142,"Jan 9, 2023",MFRM,Mattress Firm Group Inc.,-,-,Inc,NaN
143,"Sep 2, 2022",CHO,Chobani Inc.,-,-,Inc,NaN
144,"Aug 8, 2022",ROTK,"Rotech Healthcare Holdings, Inc.",-,-,Inc,NaN


In [10]:
ipos_withdrawals['Shares Offered'] = pd.to_numeric(ipos_withdrawals['Shares Offered'], errors='coerce')

In [11]:
ipos_withdrawals.isnull().sum()

Withdrawn Date     0
Symbol             0
Company Name       0
Price Range        0
Shares Offered    27
Company Class      0
Avg. price        30
dtype: int64

In [12]:
ipos_withdrawals.dropna(inplace=True)

In [13]:
ipos_withdrawals.shape

(114, 7)

In [14]:
ipos_withdrawals['Withdrawn Value'] = ipos_withdrawals['Shares Offered'] * ipos_withdrawals['Avg. price']

In [16]:
withdrawn_by_class = ipos_withdrawals.groupby('Company Class')['Withdrawn Value'].sum()

In [17]:
withdrawn_by_class

Company Class
Acq.Corp    3.200000e+08
Group       1.941250e+08
Holdings    4.224375e+08
Inc         4.041923e+09
Limited     9.069035e+08
Other       1.109296e+09
Name: Withdrawn Value, dtype: float64